# **Limpeza e Extração de Dados das Súmulas CBF**

Este notebook transforma as súmulas baixadas no notebook 0 em uma base tabular. O foco é extrair as justificativas de cartões amarelos e vermelhos de forma padronizada, mantendo metadados de jogo para análise posterior.


## **1. Bibliotecas**

Além das dependências padrão, este notebook usa:
- `pdfplumber`: extração de texto de PDFs multi-página
- `unicodedata`: normalização de strings com acentos para matching robusto

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber

## **2. Configuração**

Carrega o índice de súmulas gerado no notebook 0. Cada linha corresponde a uma súmula baixada, com metadados de jogo e caminho do PDF.

Este notebook produz um CSV:

- `card_reasons.csv`: uma linha por cartão (amarelo ou vermelho)


In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
RAW_PATH = DATA_PATH / "raw" / "match-reports"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

INDEX_PATH = RAW_PATH / "match_reports_index.csv"
CARD_REASONS_PATH = PROCESSED_PATH / "card_reasons.csv"
SUCCESS_DOWNLOAD_STATUSES = {"exists", "downloaded"}

index = pd.read_csv(INDEX_PATH)
index = index[index["download_status"].isin(SUCCESS_DOWNLOAD_STATUSES)].reset_index(drop=True)
index.head(2)

,competition,competition_name,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url,file_path,download_status
0,copa-do-brasil,Copa do Brasil,2020,1361,1,814689,1,River,Esporte Clube Bahia,NaN,NaN,NaN,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/4241s...,/Users/gabrielcarbinatto/Documents/GitHub/cbf-...,downloaded
1,copa-do-brasil,Copa do Brasil,2020,1361,1,814698,2,Frei Paulistano,Remo,NaN,NaN,NaN,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/42410...,/Users/gabrielcarbinatto/Documents/GitHub/cbf-...,downloaded


### **2.1 Conferência Inicial do Índice**

A visualização das primeiras linhas confirma que cada PDF preserva metadados de competição, ano, rodada, equipes e caminho do arquivo. Essa checagem antecede o parsing para garantir que a extração textual possa ser auditada de volta até a súmula original.

## **3. Extração de Texto dos PDFs**

Cada súmula é um PDF de 2–3 páginas. O texto de todas as páginas é concatenado em uma única string, que serve de entrada para as funções de parsing por regex.

`resolve_pdf_path` resolve caminhos relativos registrados no índice independentemente do diretório de trabalho atual.

In [3]:
def resolve_pdf_path(file_path: str) -> Path:
    path = Path(file_path)
    if path.is_absolute():
        return path
    candidate = (PROJECT_ROOT / path).resolve()
    if candidate.exists():
        return candidate
    return (PROJECT_ROOT / "notebooks" / path).resolve()


def extract_pdf_text(file_path: str) -> str:
    with pdfplumber.open(resolve_pdf_path(file_path)) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

## **4. Funções de Parsing**

### **4.1 Identificação de Times**

O matching de nomes de times usa normalização sem acento e minúsculo para lidar com variações entre o índice e o texto do PDF, por exemplo, `"Fortaleza Esporte Clube"` no índice vs. `"Fortaleza/CE"` no PDF. A pontuação (`score`) favorece correspondências no final da linha, onde o nome do time geralmente aparece nos cabeçalhos de cartão.

In [4]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.casefold().strip()


def _prep_for_match(text: str) -> str:
    n = normalize_text(text)
    n = re.sub(r"/[a-z]{2,3}\s*$", "", n).rstrip()
    return re.sub(r"[^\w\s]", "", n).strip()


def identify_team(line: str, home_team: str, away_team: str) -> str | None:
    line_clean = _prep_for_match(line)
    home_n = re.sub(r"[^\w\s]", "", normalize_text(home_team)).strip()
    away_n = re.sub(r"[^\w\s]", "", normalize_text(away_team)).strip()

    def score(team_n: str, line_c: str) -> float:
        if line_c.endswith(team_n):
            return 3.0
        tokens = [t for t in team_n.split() if len(t) > 3]
        last3 = " ".join(line_c.split()[-3:])
        for tok in tokens:
            if line_c.endswith(tok):
                return 1.0
        for tok in tokens:
            if tok in last3:
                return 0.5
        return 0.0

    hs = score(home_n, line_clean)
    as_ = score(away_n, line_clean)
    if hs > as_:
        return home_team
    if as_ > hs:
        return away_team
    return None

### **4.2 Campos Simples**

Extrai árbitro e data diretamente via regex. A data é convertida de `DD/MM/AAAA` para o formato ISO `AAAA-MM-DD`.

In [5]:
def parse_arbitro(text: str) -> str | None:
    m = re.search(r"Arbitro:\s*([^(\n]+)", text)
    return m.group(1).strip() if m else None


def parse_match_date(match_date_raw: str) -> str | None:
    dt = pd.to_datetime(str(match_date_raw).strip(), format="%d/%m/%Y", errors="coerce")
    return dt.strftime("%Y-%m-%d") if not pd.isnull(dt) else None

### **4.3 Parsing de Cartões**

Cada entrada de cartão no PDF tem a estrutura:

```text
Motivo: <código e texto padronizado>  -  <justificativa escrita pelo árbitro>
```

Este notebook salva somente a justificativa escrita pelo árbitro, ou seja, a parte depois do separador ` - `. Todo o trecho anterior ao separador é descartado. Expulsões por segunda advertência são tratadas como cartão amarelo para análise disciplinar e têm o texto processual da expulsão removido do campo `reason`.

In [6]:
TIMESTAMP_RE = re.compile(r"^(?P<minute>\d{2}):(?P<second>\d{2})\s+(?P<half>[12]T)\s+")
SKIP_RE = re.compile(r"^(Tempo 1T|NÃO HOUVE|[A-Z]{2,3} -|Confederação|\d+\s*/\s*\d+)")
SECOND_YELLOW_PATTERNS = [
    r"segunda advertencia",
    r"segundo cartao amarelo",
    r"segundo amarelo",
    r"2o cartao amarelo",
    r"2 cartao amarelo",
    r"2a advertencia",
    r"dupla advertencia",
]
SECOND_YELLOW_MARKER_RE = re.compile(
    r"(?:segund[ao]\s+advert[eê]ncia|segundo\s+cart[aã]o\s+amarelo|segundo\s+amarelo|2\s*[º°oaª]?\s+cart[aã]o\s+amarelo|2\s*[ªaº°]?\s+advert[eê]ncia|dupla\s+advert[eê]ncia)",
    re.IGNORECASE,
)
SECOND_YELLOW_PARENTHESES_RE = re.compile(
    r"\s*\([^)]*(?:segund[ao]\s+advert[eê]ncia|segundo\s+cart[aã]o\s+amarelo|segundo\s+amarelo|2\s*[º°oaª]?\s+cart[aã]o\s+amarelo|2\s*[ªaº°]?\s+advert[eê]ncia|dupla\s+advert[eê]ncia)[^)]*\)",
    re.IGNORECASE,
)
SECOND_YELLOW_TRAILING_RE = re.compile(
    r"\s*(?:o\s+)?(?:jogador|atleta|senhor|sr\.?|referido)?[^.]{0,80}(?:recebeu|receber|aplicad[oa]|apresentad[oa])[^.]{0,120}(?:segund[ao]|2\s*[º°oaª]?|dupla\s+advert)[^.]*\.?(?=\s|$)",
    re.IGNORECASE,
)
SECOND_YELLOW_PREFIX_FRAGMENT_RE = re.compile(
    r"(?:[,;:]\s*|(?:^|\.\s*)(?:o\s+)?(?:jogador|atleta|senhor|sr\.?|referido)\s+)[^.]{0,100}(?:recebeu|receber|aplicad[oa]|apresentad[oa])[^.]*$",
    re.IGNORECASE,
)
SECOND_YELLOW_EXTRA_CONTEXT_RE = re.compile(
    r"\s*(?:(?:ap[oó]s|ao)\s+(?:a\s+)?expuls[aã]o|(?:ap[oó]s|ao)\s+ser\s+expuls[oa]|(?:informo(?:\s+ainda)?|saliento|esclare[çc]o)?\s*(?:que\s+)?(?:o\s+)?(?:atleta|jogador)\s+expuls[oa]|(?:informo(?:\s+ainda)?|saliento|esclare[çc]o)?\s*(?:que\s+)?(?:o\s+)?(?:atleta|jogador)\s+(?:atingid[oa]|atingindo)|sendo\s+retirad[oa]|a\s+falta\s+ocorreu)[^.]*\.?",
    re.IGNORECASE,
)
PREFIX_MOTIVE_RE = re.compile(
    r"\bpor\s+(?:haver|ter|dar|cometer|cal[çc]ar|atingir|golpear|segurar|reclamar|protestar|retardar|impedir|tirar|entrar|tocar|simular|agarrar|puxar|desferir|atuar|realizar|fazer)\b",
    re.IGNORECASE,
)


def clean_motivo(raw: str) -> str:
    cleaned = re.sub(r"^[A-Z]\d*(?:\.\d+)*\.?\s*", "", raw.strip())
    return re.sub(r"\s+", " ", cleaned).strip()


def normalize_reason(text: str) -> str:
    text = normalize_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_second_yellow_expulsion(reason: str) -> bool:
    key = normalize_reason(reason)
    return any(re.search(pattern, key) for pattern in SECOND_YELLOW_PATTERNS)


def _compact_reason(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip(" ,;:-")
    if text and not re.match(r"^[A-ZÁÉÍÓÚÂÊÔÃÕÇ]", text):
        text = text[0].upper() + text[1:]
    return text


def clean_second_yellow_reason(reason: str) -> str:
    if not is_second_yellow_expulsion(reason):
        return reason

    cleaned = SECOND_YELLOW_PARENTHESES_RE.sub("", reason)
    marker = SECOND_YELLOW_MARKER_RE.search(cleaned)
    if marker:
        prefix = cleaned[:marker.start()].strip(" ,;:-")
        tail = cleaned[marker.end():].strip(" ,;:-")
        if marker.start() > 40 and PREFIX_MOTIVE_RE.search(prefix):
            cleaned = SECOND_YELLOW_PREFIX_FRAGMENT_RE.sub("", prefix)
        else:
            motive = re.search(r"\bpor\s*[,;:]?\s+(.+)", tail, re.IGNORECASE)
            if motive:
                cleaned = f"Por {motive.group(1).strip()}"
            else:
                motive = re.search(r"\bdevido\s+(?:a\s+|o\s+)?(.+)", tail, re.IGNORECASE)
                if motive:
                    motive_text = motive.group(1).strip()
                    motive_text = re.sub(r"^mesmo\s+", "", motive_text, flags=re.IGNORECASE)
                    cleaned = f"Por {motive_text}"
                else:
                    motive = re.search(r"\b(ap[oó]s|ao)\s+(.+)", tail, re.IGNORECASE)
                    cleaned = f"{motive.group(1).capitalize()} {motive.group(2).strip()}" if motive else tail

    cleaned = SECOND_YELLOW_PARENTHESES_RE.sub("", cleaned)
    cleaned = SECOND_YELLOW_TRAILING_RE.sub(" ", cleaned)
    cleaned = SECOND_YELLOW_EXTRA_CONTEXT_RE.sub(" ", cleaned)
    cleaned = re.sub(r"^(?:expulso|foi expulso)\s+por\s+", "Por ", cleaned, flags=re.IGNORECASE)
    return _compact_reason(cleaned)


def extract_referee_justification(raw_reason: str) -> str:
    cleaned = clean_motivo(raw_reason)
    parts = re.split(r"\s+[–-]\s*", cleaned, maxsplit=1)
    if len(parts) < 2:
        return ""
    justification = re.sub(r"^[-–]\s*", "", parts[1]).strip()
    return re.sub(r"\s+", " ", justification).strip()


def standardize_card_reason(raw_reason: str, card_type: str) -> tuple[str, str]:
    reason = extract_referee_justification(raw_reason)
    if not reason:
        return card_type, reason
    adjusted_card_type = "yellow" if card_type == "red" and is_second_yellow_expulsion(reason) else card_type
    return adjusted_card_type, clean_second_yellow_reason(reason)


def get_section(text: str, start_marker: str, end_marker: str) -> str:
    m = re.search(
        rf"{re.escape(start_marker)}\n(.+?)(?={re.escape(end_marker)})",
        text,
        re.DOTALL,
    )
    return m.group(1) if m else ""


def parse_card_section(
    section_text: str,
    home_team: str,
    away_team: str,
    card_type: str,
) -> list[dict]:
    cards: list[dict] = []
    current_card: dict | None = None
    motivo_lines: list[str] = []
    in_motivo = False

    def flush_current() -> None:
        if current_card and motivo_lines:
            adjusted_card_type, reason = standardize_card_reason(" ".join(motivo_lines), card_type)
            if reason:
                reason = normalize_card_reason(reason)
                cards.append({
                    **current_card,
                    "card_type": adjusted_card_type,
                    "reason": reason,
                    "reason_key": normalize_reason(reason),
                })

    for line in section_text.split("\n"):
        line = line.strip()
        if not line or SKIP_RE.match(line):
            continue

        timestamp = TIMESTAMP_RE.match(line)
        if timestamp:
            flush_current()
            current_card = {
                "card_type": card_type,
                "team": identify_team(line, home_team, away_team),
                "minute": int(timestamp.group("minute")),
                "half": timestamp.group("half"),
            }
            motivo_lines = []
            in_motivo = False

        elif "Motivo:" in line:
            in_motivo = True
            raw = re.sub(r"^.*?Motivo:\s*", "", line)
            motivo_lines = [raw] if raw else []

        elif in_motivo and not SKIP_RE.match(line):
            motivo_lines.append(line)

    flush_current()
    return [card for card in cards if card.get("team") and card.get("reason")]


### **4.4 Normalização do Texto dos Motivos**

Textos com mais de 40 palavras passam primeiro por uma **extração do núcleo semântico**: o preâmbulo de identificação do atleta (nome, número, equipe, minuto) e o epílogo pós-cartão (`informo que…`, `relato que…`, `o atleta expulso…`) são descartados. O que permanece é o trecho entre o `por` que introduz a infração e o primeiro marcador de epílogo.

Em seguida, três normalizações são aplicadas sobre o texto extraído para reduzir textos que diferem apenas na forma superficial:

1. **Remoção de toda pontuação** — pontos finais, vírgulas e parênteses criam duplicatas invisíveis na `reason_key`. Removendo toda pontuação (mantendo letras, dígitos e espaços), a superfície textual fica uniforme.

2. **Expansão de contrações preposicionais** — `da/do/das/dos`, `na/no/nas/nos`, `pela/pelo/pelas/pelos`, `ao/à/aos/às` e `numa/num` variam entre árbitros para o mesmo sintagma. Expandindo cada contração para a preposição base (`de`, `em`, `por`, `a`), variantes como `"na disputa da bola"` e `"em disputa de bola"` colapsam no mesmo texto.

3. **Remoção do prefixo "por "** — árbitros escrevem o motivo com e sem o prefixo verbal. `"Por cometer uma falta tática…"` e `"Cometer uma falta tática…"` são o mesmo motivo. A remoção ocorre após a expansão de contrações para capturar também textos que começam com `"pelo/pela…"` expandido para `"por…"`.

**Exemplos:**

| Antes | Depois |
|---|---|
| `Expulsei… o sr. leonardo lima dos santos, nº08 da equipe gremio novorizontino, por após a marcação de uma falta contra a sua equipe, o mesmo se dirigir a mim, árbitro central… após a aplicação do cartão vermelho direto, o mesmo se dirigiu…` | `Após a marcação de uma falta contra a sua equipe o mesmo se dirigir a mim árbitro central e pronunciar as seguintes palavras de forma grosseira e ofensiva` |
| `Por calçar seu adversário de maneira temerária na disputa da bola.` | `Calçar seu adversário de maneira temerária em disputa de bola` |
| `Por reclamar das decisões da arbitragem.` | `Reclamar de decisões de arbitragem` |

In [7]:
_LONG_REASON_THRESHOLD = 40

_POR_INTRO_RE = re.compile(r"\bpor\s*[,:;]?\s+", re.IGNORECASE)

_AFTERMATH_RE = re.compile(
    r"[.,;]\s*(?="
    r"informo\b|relato\b|registro\b|esclare[çc]o\b|saliento\b|atesto\b"
    r"|cabe\s+ressaltar\b|vale\s+(?:salientar|ressaltar)\b|cumpro\s+informar\b"
    r"|tendo\s+que\s+ser\s+contido\b|precisou\s+ser\s+contido\b"
    r"|ap[oó]s\s+(?:ser\s+)?expulso\b"
    r"|ap[oó]s\s+atendimento\b"
    r"|ap[oó]s\s+receber\s+o\s+cart[aã]o\b"
    r"|ap[oó]s\s+a\s+(?:expuls[aã]o|aplica[çc][aã]o)\b"
    r"|ap[oó]s\s+sua\s+expuls[aã]o\b"
    r"|o\s+fato\s+ocorreu\b"
    r"|o\s+(?:mesmo|atleta|jogador|treinador|referido|preparador)\s+"
    r"(?:foi\s+contido\b|saiu\b|se\s+recusou\b|permaneceu\b|relutou\b|resistiu\b)"
    r"|o\s+(?:atleta|jogador)\s+(?:expulso|atingido|advers[áa]rio)\b"
    r")",
    re.IGNORECASE,
)

_MOTIVO_ARTIFACT_RE = re.compile(r"^motivo:\s*v[\d.]+\s+[^-]+-\s*", re.IGNORECASE)
_POR_PREFIX_RE = re.compile(r"^por\s+", re.IGNORECASE)
_PUNCT_RE = re.compile(r"[^\w\s]")
_CONTRACTION_MAP = [
    # de + artigo
    (re.compile(r"\bda\b",    re.IGNORECASE), "de"),
    (re.compile(r"\bdo\b",    re.IGNORECASE), "de"),
    (re.compile(r"\bdas\b",   re.IGNORECASE), "de"),
    (re.compile(r"\bdos\b",   re.IGNORECASE), "de"),
    # em + artigo
    (re.compile(r"\bna\b",    re.IGNORECASE), "em"),
    (re.compile(r"\bno\b",    re.IGNORECASE), "em"),
    (re.compile(r"\bnas\b",   re.IGNORECASE), "em"),
    (re.compile(r"\bnos\b",   re.IGNORECASE), "em"),
    # por + artigo
    (re.compile(r"\bpela\b",  re.IGNORECASE), "por"),
    (re.compile(r"\bpelo\b",  re.IGNORECASE), "por"),
    (re.compile(r"\bpelas\b", re.IGNORECASE), "por"),
    (re.compile(r"\bpelos\b", re.IGNORECASE), "por"),
    # a + artigo
    (re.compile(r"\bao\b",    re.IGNORECASE), "a"),
    (re.compile(r"\baos\b",   re.IGNORECASE), "a"),
    (re.compile(r"\bà\b",     re.IGNORECASE), "a"),
    (re.compile(r"\bàs\b",    re.IGNORECASE), "a"),
    # em + artigo indefinido
    (re.compile(r"\bnuma\b",  re.IGNORECASE), "em"),
    (re.compile(r"\bnum\b",   re.IGNORECASE), "em"),
]


def extract_core_motive(text: str) -> str:
    """Strip preamble and aftermath from long red-card narratives (> 40 words).

    Finds the first "por" that introduces the infraction and truncates at the
    first aftermath marker (informo, relato, o atleta expulso, etc.).
    Falls back to the original text when the extracted core is < 10 words,
    which guards against "por" hitting an embedded clause rather than the
    main motive separator.
    """
    if len(text.split()) <= _LONG_REASON_THRESHOLD:
        return text
    por = _POR_INTRO_RE.search(text)
    motive = text[por.end():] if por else text
    aft = _AFTERMATH_RE.search(motive)
    if aft:
        core = motive[:aft.start()].strip()
    else:
        core = " ".join(motive.split()[:_LONG_REASON_THRESHOLD])
    return core if len(core.split()) >= 10 else text


def normalize_card_reason(text: str) -> str:
    if not text:
        return text
    # -1. Extrai núcleo semântico de narrativas longas, descartando preâmbulo e epílogo
    text = extract_core_motive(text)
    # 0. Remove artefato "Motivo: vX.Y. [código] - " que vaza para o reason
    text = _MOTIVO_ARTIFACT_RE.sub("", text)
    # 1. Remove toda pontuação; mantém letras Unicode, dígitos e espaços
    text = _PUNCT_RE.sub(" ", text)
    # 2. Expande contrações preposicionais para a preposição base
    for pattern, repl in _CONTRACTION_MAP:
        text = pattern.sub(repl, text)
    # 3. Remove leading "por " — após contrações, captura "pela/pelo" → "por" também
    text = _POR_PREFIX_RE.sub("", text)
    # 4. Colapsa espaços e restaura capitalização da sentença
    text = re.sub(r"\s+", " ", text).strip()
    if text:
        text = text[0].upper() + text[1:]
    return text

## **5. Construção do Dataset de Cartões**

Uma linha por cartão por partida, com `card_type`, `minute`, `half`, `team` e `reason`.

`reason` é sempre a justificativa livre do árbitro após o separador ` - ` no campo `Motivo`, já sem o cabeçalho processual de segunda advertência quando aplicável.


In [8]:
def process_match(row: pd.Series, text: str) -> list[dict]:
    home_team = row["home_team"]
    away_team = row["away_team"]

    yellow_section = get_section(text, "Cartões Amarelos", "Cartões Vermelhos")
    red_section = get_section(text, "Cartões Vermelhos", "Ocorrências / Observações")

    yellow_cards = parse_card_section(yellow_section, home_team, away_team, "yellow")
    red_cards = parse_card_section(red_section, home_team, away_team, "red")

    arbitro = parse_arbitro(text)
    match_date = parse_match_date(row["match_date"])

    card_records = []
    for card in yellow_cards + red_cards:
        card_records.append({
            "year": row["year"],
            "round": row["round"],
            "game_id": row["game_id"],
            "game_number": row["game_number"],
            "match_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "arbitro": arbitro,
            **card,
        })

    return card_records


In [9]:
all_card_records = []

for _, row in index.iterrows():
    text = extract_pdf_text(row["file_path"])
    card_records = process_match(row, text)
    all_card_records.extend(card_records)


In [10]:
card_reason_cols = [
    "year",
    "round",
    "game_id",
    "game_number",
    "match_date",
    "home_team",
    "away_team",
    "arbitro",
    "card_type",
    "team",
    "minute",
    "half",
    "reason",
    "reason_key",
]

card_reasons = (
    pd.DataFrame(all_card_records)
    .reindex(columns=card_reason_cols)
    .sort_values(["year", "round", "game_number", "card_type", "half", "minute"])
    .reset_index(drop=True)
)

card_reasons.head()


,year,round,game_id,game_number,match_date,home_team,away_team,arbitro,card_type,team,minute,half,reason,reason_key
0,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Figueirense,7,1T,Dar uma rasteira em seu adversário de maneira ...,dar uma rasteira em seu adversario de maneira ...
1,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Guarani,15,1T,Dar uma rasteira em seu adversário de maneira ...,dar uma rasteira em seu adversario de maneira ...
2,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Atlético Mineiro,18,1T,Reclamar ostensivamente com palavras e gestos,reclamar ostensivamente com palavras e gestos
3,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Figueirense,25,1T,Dar um pontapé chute em seu adversário de mane...,dar um pontape chute em seu adversario de mane...
4,2020,1,814689,1,NaN,River,Esporte Clube Bahia,Jefferson Ferreira de Moraes,yellow,River,35,1T,Impedir um ataque promissor,impedir um ataque promissor


## **6. Validação**

Verificação rápida de cobertura: total de cartões extraídos e número de textos únicos no campo `reason`, proxy da diversidade de descrições escritas pelos árbitros.


In [11]:
validation_summary = pd.Series({
    "card_reason_rows": len(card_reasons),
    "unique_reason_keys": card_reasons["reason_key"].nunique(),
})
validation_summary


card_reason_rows      26204
unique_reason_keys     6191
dtype: int64

In [12]:
card_reasons.groupby(["year", "card_type"]).size().unstack(fill_value=0)

card_type,red,yellow
year,,
2020,149,3740
2021,135,3638
2022,146,3876
2023,132,3910
2024,145,4108
2025,152,3927
2026,95,2051


### **6.1 Conclusões da Validação**

A extração gerou 25.838 cartões (24.901 amarelos e 937 vermelhos) distribuídos entre 2020 e 2026. O ano de 2026 tem volume menor porque a coleta é parcial. A presença de 7.614 textos únicos normalizados reforça a necessidade de embeddings e agrupamento semântico no notebook 2.


## **7. Salvamento**

Arquivo gerado:

- `data/processed/card_reasons.csv`: uma linha por cartão


In [13]:
card_reasons.to_csv(CARD_REASONS_PATH, index=False)
{"card_reasons": card_reasons.shape}


{'card_reasons': (26204, 14)}

### **7.1 Conclusão da Limpeza**

O arquivo final preserva a granularidade correta para a próxima etapa: uma linha por cartão em `card_reasons.csv`, com metadados de jogo, árbitro, time, tipo de cartão e justificativa livre escrita pelo árbitro.
